# regime 1 / Three-factor **v3** — cos ≈ 0.01 (from-scratch ViT, MNIST)
The extreme of the cos sweep. `TARGET_COS=0.01` picks **M ≈ 100 probes/step** (vs v1's ~337k for cos=0.5, v2's ~8k for cos≈0.09), so each gradient is *very* noisy but each step is dirt-cheap — this run takes the **most** steps of the three. Tests whether the many-noisier-steps trend keeps winning or finally breaks. Same model (P≈1M), seed, data and equal-wall-clock harness as the other regime-1 methods; per-minute heartbeat; writes `results/three_factor_v3.json`.

## Step 0 — Setup

In [ ]:
import os, time, math, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: no GPU. Built for an A100 Colab runtime; set SMOKE=True to sanity-check on CPU.')

## Config (v3 — cos≈0.01)

In [ ]:
METHOD = 'three_factor_v3'
SIGMA = 1e-2
TARGET_COS = 0.01   # v3: pick M so cos(g_hat, grad) ≈ this (cos^2 = M/(M+P+1)) -> ~100 probes at P≈1M
CHUNK_PROBES = 512
LR_LOCAL = 0.05
MOMENTUM = 0.9
CLIP_GRAD_NORM = 1.0
HEARTBEAT_EVERY_SEC = 60   # compact status line this often (v3 is very fast per step)
METHOD_CFG = dict(SIGMA=SIGMA, TARGET_COS=TARGET_COS, CHUNK_PROBES=CHUNK_PROBES, LR_LOCAL=LR_LOCAL, MOMENTUM=MOMENTUM, CLIP_GRAD_NORM=CLIP_GRAD_NORM)


# =============================== CONFIG (shared across all 3 methods) ===============================
# Architecture -- IDENTICAL for backprop / two-factor / three-factor (the only fair race). P ~ 1.01M.
IMG, PATCH, IN_CH, NUM_CLASSES = 28, 7, 1, 10
EMBED_DIM, DEPTH, HEADS, MLP_RATIO = 176, 4, 8, 2

# Fair match = EQUAL WALL-CLOCK. Each method trains until this budget, then reports. Run the 3
# notebooks simultaneously on 3 A100 runtimes; compare with 04_compare_results.ipynb.
TIME_BUDGET_HOURS = 5.0
BATCH             = 64          # images per gradient step (NOT the whole 60k train set; 1 epoch = 938 steps)
SEED              = 0           # same init across all three methods
EVAL_BATCH        = 2000

# Cadences (wall-clock seconds)
LOG_EVERY_SEC   = 30           # record a (time, loss, acc) point
CKPT_EVERY_SEC  = 120          # save a checkpoint (survives Colab disconnects)
PLOT_EVERY_SEC  = 3600         # redraw the live loss-vs-time graph (default: hourly)

# Storage: Google Drive so the 3 runtimes share results and checkpoints survive disconnects.
USE_DRIVE   = True
DRIVE_SUBDIR = 'Section8_regime1'

PLOT_EVERY_SEC = 1800   # full graph every 30 min; heartbeat covers the minute view

# Fast end-to-end validation (tiny model, ~20s budget, no Drive) -- flip to False for the real run.
SMOKE = False
if SMOKE:
    EMBED_DIM, DEPTH, HEADS = 32, 1, 2
    TIME_BUDGET_HOURS = 20/3600
    LOG_EVERY_SEC, CKPT_EVERY_SEC, PLOT_EVERY_SEC = 2, 5, 8
    EVAL_BATCH, USE_DRIVE = 1000, False
# ===================================================================================================

if SMOKE: HEARTBEAT_EVERY_SEC = 1

## Step 1 — Storage (Drive) + paths

In [ ]:
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        STORE = os.path.join('/content/drive/MyDrive', DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed -> local /content (will NOT survive disconnect):', e)
        STORE = os.path.join('/content', DRIVE_SUBDIR)
else:
    STORE = os.path.join('.', DRIVE_SUBDIR)
RESULTS_DIR = os.path.join(STORE, 'results')
CKPT_DIR    = os.path.join(STORE, 'checkpoints')
FIG_DIR     = os.path.join(STORE, 'figures')
for d in (RESULTS_DIR, CKPT_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
RESULTS_PATH = os.path.join(RESULTS_DIR, f'{METHOD}.json')
CKPT_PATH    = os.path.join(CKPT_DIR,    f'{METHOD}.pt')
print('storing under:', STORE)

## Step 2 — Data (MNIST)

In [ ]:
import torchvision
train_ds = torchvision.datasets.MNIST('./data', train=True,  download=True)
test_ds  = torchvision.datasets.MNIST('./data', train=False, download=True)
mean, std = 0.1307, 0.3081
Xtr = ((train_ds.data.float()/255.0 - mean)/std).unsqueeze(1).to(device)
Ytr = train_ds.targets.to(device)
Xte = ((test_ds.data.float()/255.0 - mean)/std).unsqueeze(1).to(device)
Yte = test_ds.targets.to(device)
print('MNIST train', tuple(Xtr.shape), '| test', tuple(Xte.shape))

def fresh_batch(n):
    idx = torch.randint(0, Xtr.shape[0], (n,), device=device)
    return Xtr[idx], Ytr[idx]

## Step 3 — The ViT

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F

class MHSA(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        assert dim % heads == 0, "EMBED_DIM must be divisible by HEADS"
        self.h, self.dh = heads, dim // heads
        self.scale = self.dh ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q @ k.transpose(-2, -1)) * self.scale
        att = att.softmax(dim=-1)
        o = (att @ v).transpose(1, 2).reshape(B, N, D)
        return self.proj(o)

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = MHSA(dim, heads)
        self.n2 = nn.LayerNorm(dim)
        h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img=28, patch=7, in_ch=1, dim=176, depth=4, heads=8,
                 mlp_ratio=2, num_classes=10):
        super().__init__()
        assert img % patch == 0
        self.n = (img // patch) ** 2
        self.patch = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, self.n + 1, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.cls, std=0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch(x).flatten(2).transpose(1, 2)      # (B, n, dim)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)[:, 0]
        return self.head(x)

## Step 4 — Build + provenance (M from target cos)

In [ ]:
torch.manual_seed(SEED)
model = ViT(IMG, PATCH, IN_CH, EMBED_DIM, DEPTH, HEADS, MLP_RATIO, NUM_CLASSES).to(device)
P = sum(p.numel() for p in model.parameters())
NUM_TOK = (IMG // PATCH)**2 + 1
Mstar = math.ceil((P + 1) / 3)
M = max(1, round(TARGET_COS**2 * (P + 1) / (1 - TARGET_COS**2)))   # M chosen for the target per-step cos
got = (M / (M + P + 1))**0.5
print(f'METHOD = {METHOD} | P = {P:,} | tokens/img = {NUM_TOK}')
print(f'target cos = {TARGET_COS} -> M = {M:,} probes/step (actual cos = {got:.4f}); M*(cos=0.5) = {Mstar:,}')
print('v3 is the extreme point: very few probes -> very noisy per-step gradient -> the MOST steps.')
CFG = dict(METHOD=METHOD, SEED=SEED, PATCH=PATCH, EMBED_DIM=EMBED_DIM, DEPTH=DEPTH, HEADS=HEADS,
           MLP_RATIO=MLP_RATIO, BATCH=BATCH, TIME_BUDGET_HOURS=TIME_BUDGET_HOURS)
CFG.update(METHOD_CFG); CFG['M'] = M

## Step 5 — Probe estimator (vmap antithetic)

In [ ]:
from torch.func import functional_call, vmap

base_buffers = {k: v.detach() for k, v in model.named_buffers()}

def get_params(m):
    return {k: v.detach().clone() for k, v in m.named_parameters()}

def loss_at(params, x, y):
    logits = functional_call(model, (params, base_buffers), (x,))
    return F.cross_entropy(logits, y)

_vloss = vmap(loss_at, in_dims=(0, None, None))

def estimate_grad(params, x, y, M, sigma, chunk):
    """Antithetic zeroth-order (three-factor) estimate of grad L, averaged over M probes,
    computed in vmap-batched chunks. E[estimate] = grad of the Gaussian-smoothed loss."""
    g = {k: torch.zeros_like(v) for k, v in params.items()}
    done = 0
    while done < M:
        m = min(chunk, M - done)
        xi = {k: torch.randn((m,) + v.shape, device=v.device, dtype=v.dtype)
              for k, v in params.items()}
        pp = {k: params[k].unsqueeze(0) + sigma * xi[k] for k in params}
        pm = {k: params[k].unsqueeze(0) - sigma * xi[k] for k in params}
        score = (_vloss(pp, x, y) - _vloss(pm, x, y)) / (2 * sigma)   # (m,)
        for k in params:
            g[k] += torch.tensordot(score, xi[k], dims=([0], [0]))
        done += m
    return {k: g[k] / M for k in g}

## Step 6 — Trainer (three-factor)

In [ ]:
params   = get_params(model)                       # seeded init, identical to the other two methods
velocity = {k: torch.zeros_like(v) for k, v in params.items()}

def forward_fn(x): return functional_call(model, (params, base_buffers), (x,))

def train_step():
    xb, yb = fresh_batch(BATCH)
    g = estimate_grad(params, xb, yb, M, SIGMA, CHUNK_PROBES)     # vmap antithetic estimator
    if CLIP_GRAD_NORM:                                            # global-norm clip (long-run safety)
        gn = math.sqrt(sum(float((v * v).sum()) for v in g.values()))
        if gn > CLIP_GRAD_NORM:
            s = CLIP_GRAD_NORM / (gn + 1e-12)
            for k in g: g[k].mul_(s)
    for k in params:
        velocity[k] = MOMENTUM * velocity[k] + g[k]
        params[k]  -= LR_LOCAL * velocity[k]
    with torch.no_grad():
        return F.cross_entropy(forward_fn(xb), yb).item()

def ckpt_state():      return {'params': params, 'velocity': velocity}
def load_ckpt_state(d):
    for k in params: params[k].copy_(d['params'][k]); velocity[k].copy_(d['velocity'][k])

## Step 7 — Eval / checkpoint / live-plot

In [ ]:
@torch.no_grad()
def eval_metrics():
    correct = tot = 0; loss_sum = 0.0
    for i in range(0, Xte.shape[0], EVAL_BATCH):
        logits = forward_fn(Xte[i:i+EVAL_BATCH]); yb = Yte[i:i+EVAL_BATCH]
        correct += (logits.argmax(-1) == yb).sum().item(); tot += yb.numel()
        loss_sum += F.cross_entropy(logits, yb, reduction='sum').item()
    return loss_sum/tot, correct/tot

In [ ]:
def _atomic_save(obj, path):
    tmp = path + '.tmp'; torch.save(obj, tmp); os.replace(tmp, path)

def save_checkpoint(step, elapsed, logs):
    _atomic_save({'loop': {'step': step, 'elapsed_sec': elapsed, 'logs': logs},
                  'method': ckpt_state(), 'config': CFG}, CKPT_PATH)

def maybe_resume():
    if os.path.exists(CKPT_PATH):
        d = torch.load(CKPT_PATH, map_location=device)
        load_ckpt_state(d['method']); L = d['loop']
        print(f'[resume] {METHOD}: step {L["step"]:,}, elapsed {L["elapsed_sec"]/3600:.2f}h -> continuing.')
        return L['step'], L['elapsed_sec'], L['logs']
    return 0, 0.0, []

In [ ]:
def live_plot(logs, final=False):
    if not logs: return
    hrs = [l[0]/3600 for l in logs]
    tr  = [l[3] for l in logs]; te = [l[4] for l in logs]; ac = [l[5] for l in logs]
    if not final: clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hrs, tr, '-',  color='#999', label='train loss')
    ax[0].plot(hrs, te, 'o-', color='#C62828', label='test loss')
    ax[0].set_xlabel('elapsed hours'); ax[0].set_ylabel('loss')
    ax[0].set_title(f'{METHOD}: loss vs time'); ax[0].legend()
    ax[1].plot(hrs, ac, 'o-', color='#1F3864')
    ax[1].set_xlabel('elapsed hours'); ax[1].set_ylabel('test accuracy'); ax[1].set_title('accuracy vs time')
    plt.tight_layout()
    try: fig.savefig(os.path.join(FIG_DIR, f'{METHOD}_progress.png'), dpi=90)
    except Exception as e: print('savefig failed:', e)
    plt.show()
    last = logs[-1]
    print(f'[h={last[0]/3600:.2f}] step={last[1]:,} epoch={last[2]:.3f} test_loss={last[4]:.4f} acc={last[5]:.3f}')

## Step 8 — Train (equal wall-clock, with heartbeat) + save results

In [ ]:
def write_results(step, elapsed, logs):
    te = [l[4] for l in logs]; ac = [l[5] for l in logs]
    summary = dict(initial_loss=logs[0][4], final_loss=logs[-1][4], best_loss=min(te),
                   reduction_pct=100*(logs[0][4]-min(te))/max(logs[0][4], 1e-9),
                   final_acc=logs[-1][5], best_acc=max(ac))
    peak = torch.cuda.max_memory_allocated()/1e6 if device == 'cuda' else float('nan')
    out = dict(meta=dict(method=METHOD, device=device, seed=SEED, P=P, config=CFG,
                         note='all MEASURED-here', wall_clock_sec=elapsed, total_steps=step,
                         epochs=step*BATCH/60000, peak_mem_mb=peak),
               curve=dict(t_sec=[l[0] for l in logs], step=[l[1] for l in logs], epoch=[l[2] for l in logs],
                          train_loss=[l[3] for l in logs], test_loss=te, test_acc=ac),
               summary=summary)
    with open(RESULTS_PATH, 'w') as f: json.dump(out, f, indent=2)
    print('wrote', RESULTS_PATH)

def run():
    step, elapsed0, logs = maybe_resume()
    t0 = time.time(); budget = TIME_BUDGET_HOURS*3600
    last_ckpt = last_plot = last_log = last_beat = elapsed0
    if not logs:
        l0, a0 = eval_metrics(); logs.append((elapsed0, step, 0.0, l0, l0, a0))
    tl = logs[-1][3]; step1_printed = False
    while True:
        elapsed = elapsed0 + (time.time() - t0)
        if elapsed >= budget: break
        tl = train_step(); step += 1
        if not step1_printed:
            dt = time.time() - t0
            print(f'[{METHOD}] step 1 = {dt:.2f}s -> ~{int(budget/max(dt,1e-9)):,} steps fit the budget.', flush=True)
            step1_printed = True
        if elapsed - last_log >= LOG_EVERY_SEC:
            el, ac = eval_metrics(); logs.append((elapsed, step, step*BATCH/60000, tl, el, ac)); last_log = elapsed
        if elapsed - last_beat >= HEARTBEAT_EVERY_SEC:
            L = logs[-1]
            print(f'[{METHOD}] h={elapsed/3600:.2f} step={step:,} train={L[3]:.4f} test={L[4]:.4f} acc={L[5]:.3f}', flush=True)
            last_beat = elapsed
        if elapsed - last_ckpt >= CKPT_EVERY_SEC:
            save_checkpoint(step, elapsed, logs); last_ckpt = elapsed
        if elapsed - last_plot >= PLOT_EVERY_SEC:
            live_plot(logs); last_plot = elapsed
    elapsed = elapsed0 + (time.time() - t0)
    el, ac = eval_metrics(); logs.append((elapsed, step, step*BATCH/60000, tl, el, ac))
    save_checkpoint(step, elapsed, logs); live_plot(logs, final=True); write_results(step, elapsed, logs)
    print(f'DONE {METHOD}: {step:,} steps, {elapsed/3600:.2f}h, final test acc {ac:.3f}, best loss {min(l[4] for l in logs):.4f}')

run()

## How to run
- **`SMOKE=True` first**, then `False` → Runtime → Run all on an A100.
- Heartbeat every ~60 s; disconnect-safe (own `three_factor_v3.pt`, auto-resume).
- Because the per-step gradient is so noisy (cos≈0.01), **`MOMENTUM` and `LR_LOCAL` matter most** — if it stalls or diverges, lower `LR_LOCAL` (e.g. 0.02) or raise `MOMENTUM` (0.95). Bump `TARGET_COS` back up (0.03–0.05) if cos=0.01 is too noisy to learn.
- Compare v1/v2/v3 in `04_compare_results.ipynb` (§7 head-to-head).